# Wedding announcement text, the sanctioned route

The Upshot's weddings CSV is deliberately metadata-only: the articles themselves are
paywalled, copyrighted NYT journalism, and scraping nytimes.com violates its terms of
service — the very line this course teaches in Week 4 (and litigates in Week 7's
reading). The sanctioned route to real announcement *text* is the official **NYT Article
Search API**: a free key from developer.nytimes.com returns each article's headline,
abstract, and **lead paragraph** — and for announcements, the lead paragraph carries the
formula (the parents, the colleges, the occupations) that makes this corpus so studied.

**Licensing tier: analyze, don't redistribute.** Count and publish findings; never
republish the text or share the collected file. Full text is not available by any
sanctioned bulk route; the lead paragraph is what there is, and it is enough for real
questions.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder — Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
import os, io, time, re
import pandas as pd

NYT_KEY = os.environ.get("NYT_API_KEY")     # free key: developer.nytimes.com -> Colab secret
try:
    from google.colab import userdata
    NYT_KEY = NYT_KEY or userdata.get("NYT_API_KEY")
except Exception:
    pass
LIVE = bool(NYT_KEY)

# A clearly synthetic sample in the genre's shape, so the pipeline reads offline.
# (Names are placeholders, not people.)
RECORDED = [{
    "web_url": "https://www.nytimes.com/1995/06/11/style/example.html",
    "headline": "A. Placeholder Weds B. Example",
    "lead_paragraph": ("Ann Placeholder, a daughter of Mr. and Mrs. R. Placeholder of "
                       "Scarsdale, N.Y., was married yesterday to Brian Example, a son of "
                       "Dr. and Mrs. T. Example of Boston. The bride, 27, is an associate "
                       "at a Manhattan law firm; she graduated from Cornell. The bridegroom, "
                       "29, an editor, graduated from Amherst."),
    "pub_date": "1995-06-11"}]

## Query a month of announcements

The Article Search API filters by date and section. Respect the free tier's limits: a
few requests per minute and a daily cap, so put a long pause between pages and collect
over days, not minutes, for the full thirty years. One month is plenty to start asking
questions.

In [ ]:
API = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

def month_of_announcements(begin="19950601", end="19950630", pages=3):
    """Lead paragraphs for one month of Style-section wedding items; offline, the sample."""
    if not LIVE:
        print("no NYT_API_KEY set, using the synthetic sample (pipeline is identical)")
        return pd.DataFrame(RECORDED)
    import requests
    rows = []
    for page in range(pages):
        r = requests.get(API, timeout=20, params={
            "api-key": NYT_KEY, "begin_date": begin, "end_date": end,
            "fq": 'section_name:("Style") AND headline:("wed" "weds" "married" "bride")',
            "page": page})
        r.raise_for_status()
        for d in r.json()["response"]["docs"]:
            rows.append({"web_url": d.get("web_url"),
                         "headline": (d.get("headline") or {}).get("main"),
                         "lead_paragraph": d.get("lead_paragraph"),
                         "pub_date": (d.get("pub_date") or "")[:10]})
        time.sleep(12)   # the free tier is a few requests per minute; this pause is the rule
    return pd.DataFrame(rows)

texts = month_of_announcements()
print(len(texts), "announcements with text")
print(texts.iloc[0]["lead_paragraph"][:200], "...")

## What the lead paragraphs can carry

Even this slice supports real questions: which occupations appear, whose parents are
named ("a daughter of" versus "a son of" constructions), which colleges recur, when
"Miss" disappears. The same counting moves as Week 1, on the genre's actual sentences.

In [ ]:
words = [w for t in texts["lead_paragraph"].dropna()
         for w in re.findall(r"[a-z]+", t.lower())]
from collections import Counter
STOP = {"the","of","a","an","to","is","in","for","and","on","with","at","was","her","his","she","he","mr","mrs","ms"}
print("top words:", Counter(w for w in words if w not in STOP and len(w) > 2).most_common(12))

# Join back to the Upshot rows by URL to combine text with the name-keeping field:
# merged = texts.merge(weddings, left_on="web_url", right_on="url", how="inner")

**Collecting more.** The daily cap means the full 1985–2014 corpus is a multi-day,
save-as-you-go collection: one month per session, appended to a CSV in your Drive project
folder. That patience is not an obstacle to route around; it is what using data on its
owner's terms looks like. Consult developer.nytimes.com for the current limits and terms.